# PCB Image Generation
ControlNet + LoRA inference from a pre-existing structure map.

**Workflow:**
1. Set parameters in **Cell 2**
2. Run **Cell 3** once to load models
3. Run **Cell 4** as many times as needed to generate

In [1]:
from __future__ import annotations
import os, sys
from pathlib import Path

import env
os.environ["HF_HOME"] = env.HF_HOME

import cv2
import numpy as np
import torch
from diffusers import (
    ControlNetModel,
    StableDiffusionXLControlNetImg2ImgPipeline,
    StableDiffusionXLControlNetPipeline,
)
from peft import PeftModel
from PIL import Image
import matplotlib.pyplot as plt

/Users/alexblokh/study/pcb-generation/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT = Path.cwd()

# ── Model paths ───────────────────────────────────────────────────────────────
CONTROLNET_PATH = ROOT / "trained/controlnet_aug_600x600"
LORA_PATH       = ROOT / "trained/lora_aug_600x600"

# ── Input / output ────────────────────────────────────────────────────────────
STRUCTURE_MAP_PATH = ROOT / "images/structure_maps/layout_2.png"
OUTPUT_DIR         = ROOT / "images/generated_debug"

# ── Prompts ───────────────────────────────────────────────────────────────────
PROMPT = (
    "macro photo of printed circuit board, green solder mask, "
    "copper traces, vias, electronic components, realistic, photorealistic"
)
NEGATIVE_PROMPT = "blurry, low quality, defects, damage, cracks, burned"

# ── Generation parameters ─────────────────────────────────────────────────────
HEIGHT              = 600
WIDTH               = 600
NUM_INFERENCE_STEPS = 50
GUIDANCE_SCALE      = 8.0
CONTROLNET_SCALE    = 0.8
LORA_SCALE          = 0.85
REFINE_STRENGTH     = 0.25   # img2img refinement strength (0 = no change, 1 = full re-diffuse)
SEED                = 42

In [3]:
# ── Device setup ──────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
weight_dtype = torch.float16 if device.type == "cuda" else torch.float32
print(f"Device: {device}  |  dtype: {weight_dtype}")

# ── ControlNet ────────────────────────────────────────────────────────────────
print("Loading ControlNet...")
controlnet = ControlNetModel.from_pretrained(
    str(CONTROLNET_PATH), torch_dtype=weight_dtype
)

# ── SDXL + LoRA ───────────────────────────────────────────────────────────────
print("Loading SDXL + LoRA...")
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    controlnet=controlnet,
    torch_dtype=weight_dtype,
    low_cpu_mem_usage=True,
)
pipe.unet = PeftModel.from_pretrained(pipe.unet, str(LORA_PATH))
for module in pipe.unet.modules():
    if hasattr(module, "scaling"):
        for key in module.scaling:
            module.scaling[key] = LORA_SCALE
pipe.unet.eval()

if device.type != "mps":
    pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()
pipe.enable_attention_slicing()

# ── Refinement pipeline (shares loaded weights, no extra VRAM) ────────────────
print("Building img2img refinement pipeline...")
pipe_i2i = StableDiffusionXLControlNetImg2ImgPipeline(**pipe.components)

print("\nPipelines ready.")

Device: mps  |  dtype: torch.float32
Loading ControlNet...
Loading SDXL + LoRA...


Loading pipeline components...: 100%|██████████| 7/7 [00:00<00:00, 21.63it/s]


Building img2img refinement pipeline...

Pipelines ready.


/Users/alexblokh/study/pcb-generation/venv/lib/python3.11/site-packages/diffusers/pipelines/pipeline_utils.py:2263: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `StableDiffusionXLControlNetPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(


In [4]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
stem = Path(STRUCTURE_MAP_PATH).stem

# ── Load structure map ────────────────────────────────────────────────────────
structure_np = cv2.imread(str(STRUCTURE_MAP_PATH), cv2.IMREAD_GRAYSCALE)
if structure_np is None:
    raise FileNotFoundError(f"Structure map not found: {STRUCTURE_MAP_PATH}")
structure_np = cv2.resize(structure_np, (WIDTH, HEIGHT))
structure_pil = Image.fromarray(structure_np)

# MPS doesn't support on-device generators in diffusers — always use CPU
gen_device = "cpu" if device.type == "mps" else device.type

# ── Step 1: ControlNet text → image ──────────────────────────────────────────
print("Generating raw image...")
gen = torch.Generator(device=gen_device).manual_seed(SEED)
raw_pil = pipe(
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    image=structure_pil,
    num_inference_steps=NUM_INFERENCE_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    controlnet_conditioning_scale=CONTROLNET_SCALE,
    height=HEIGHT,
    width=WIDTH,
    generator=gen,
).images[0]
raw_pil.save(OUTPUT_DIR / f"{stem}_raw.png")

# ── Step 2: img2img refinement ────────────────────────────────────────────────
# print("Refining...")
# gen2 = torch.Generator(device=gen_device).manual_seed(SEED)
# final_pil = pipe_i2i(
#     prompt=PROMPT,
#     negative_prompt=NEGATIVE_PROMPT,
#     image=raw_pil,
#     control_image=structure_pil,
#     strength=REFINE_STRENGTH,
#     num_inference_steps=NUM_INFERENCE_STEPS,
#     guidance_scale=GUIDANCE_SCALE,
#     controlnet_conditioning_scale=CONTROLNET_SCALE,
#     generator=gen2,
# ).images[0]
# final_pil.save(OUTPUT_DIR / f"{stem}_final.png")

# # ── Display ───────────────────────────────────────────────────────────────────
# fig, axes = plt.subplots(1, 3, figsize=(15, 5))
# axes[0].imshow(structure_pil, cmap="gray"); axes[0].set_title("Structure Map");  axes[0].axis("off")
# axes[1].imshow(raw_pil);                    axes[1].set_title("Raw Generation"); axes[1].axis("off")
# axes[2].imshow(final_pil);                  axes[2].set_title("Refined");        axes[2].axis("off")
# plt.tight_layout()
# plt.show()

# print(f"Saved → {OUTPUT_DIR / f'{stem}_raw.png'}")
# print(f"Saved → {OUTPUT_DIR / f'{stem}_final.png'}")

Generating raw image...


 16%|█▌        | 8/50 [05:03<26:35, 37.99s/it]


KeyboardInterrupt: 